# Week 6 — Data Representation & Model Evaluation

**Part 1** covers representing data well (encoding, feature engineering,
missing values, outliers, leakage-safe scaling). **Part 2** covers
evaluating a model honestly once you have it (cross-validation, confusion
matrix, precision/recall, ROC/AUC) — the natural next question once your
data is in good shape.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

## 1. Build a synthetic used-car dataset

We generate 300 used car listings with a known (but noisy) pricing rule, so we
can see clearly how much each data-representation trick helps.

In [ ]:
rng = np.random.default_rng(7)
n = 300

brands = rng.choice(["Hyundai", "Kia", "Toyota", "BMW"], size=n, p=[0.35, 0.3, 0.2, 0.15])
fuel_types = rng.choice(["gasoline", "diesel", "electric"], size=n, p=[0.55, 0.3, 0.15])
transmissions = rng.choice(["automatic", "manual"], size=n, p=[0.8, 0.2])
purchase_year = rng.integers(2010, 2024, size=n)
mileage_km = rng.integers(5_000, 180_000, size=n)

brand_premium = {"Hyundai": 0, "Kia": 0, "Toyota": 3_000, "BMW": 12_000}
fuel_premium = {"gasoline": 0, "diesel": 1_000, "electric": 6_000}

age = 2024 - purchase_year
base_price = 28_000
price = (
    base_price
    - age * 1_400                 # older car -> cheaper
    - mileage_km * 0.05           # more mileage -> cheaper
    + np.array([brand_premium[b] for b in brands])
    + np.array([fuel_premium[f] for f in fuel_types])
    + rng.normal(0, 1_500, size=n)  # noise: negotiation, condition, luck
)
price = np.clip(price, 2_000, None)

cars = pd.DataFrame({
    "brand": brands,
    "fuel_type": fuel_types,
    "transmission": transmissions,
    "purchase_year": purchase_year,
    "mileage_km": mileage_km,
    "price": price.round(0),
})
cars.head()

## Step 1: Numbers-only baseline (ignore the categories)

The naive approach: drop every text column and use only the numeric ones.

In [ ]:
def evaluate(X, y, label):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
    model = LinearRegression().fit(Xtr, ytr)
    mae = mean_absolute_error(yte, model.predict(Xte))
    print(f"{label:35s} MAE = ${mae:,.0f}")
    return mae

results = {}
X1 = cars[["purchase_year", "mileage_km"]]
results["1. numeric only"] = evaluate(X1, cars["price"], "1. Numeric columns only")

## Step 2: One-hot encode the categorical columns

`pd.get_dummies` turns each category into its own 0/1 column, so "brand" (a
word) becomes 4 numeric columns the model can use — without implying any order
between brands.

In [ ]:
categorical_encoded = pd.get_dummies(cars[["brand", "fuel_type", "transmission"]])
categorical_encoded.head()

In [ ]:
X2 = pd.concat([cars[["purchase_year", "mileage_km"]], categorical_encoded], axis=1)
results["2. + one-hot categories"] = evaluate(X2, cars["price"], "2. + one-hot encoded categories")

## Step 3: Feature engineering — `age` instead of `purchase_year`

A linear model can only learn "the *bigger* this number, the *more/less* the
price changes." `purchase_year` (e.g. 2010 vs. 2023) works, but `age` (car's
age in years) is a more natural, directly-meaningful feature for the same
underlying relationship. Feature engineering means *reshaping* raw values into
a form that's easier for the model to use.

In [ ]:
cars["age"] = 2024 - cars["purchase_year"]

X3 = pd.concat([cars[["age", "mileage_km"]], categorical_encoded], axis=1)
results["3. + engineered 'age' feature"] = evaluate(X3, cars["price"], "3. + engineered 'age' feature")

## Step 4: Scale the numeric features

`mileage_km` ranges up to 180,000 while `age` maxes out around 14 — very
different scales. Some models (and especially neural networks, coming in
Week 8) train better when every feature is standardized to a similar range.

In [ ]:
numeric_cols = ["age", "mileage_km"]
scaler = StandardScaler()
X4 = X3.copy()
X4[numeric_cols] = scaler.fit_transform(X4[numeric_cols])

results["4. + scaled numeric features"] = evaluate(X4, cars["price"], "4. + scaled numeric features")

## Compare all four representations

In [ ]:
summary = pd.Series(results)
print(summary)

plt.figure(figsize=(7, 4))
summary.plot(kind="barh", color="steelblue")
plt.title("Prediction Error (MAE, lower = better) by Data Representation")
plt.xlabel("Mean Absolute Error ($)")
plt.gca().invert_yaxis()
plt.show()

## 5. Real-world practice: Titanic — missing values, outliers, leakage-safe scaling

So far we've used a synthetic used-car dataset to practice encoding and
feature engineering. Now let's use real
[Titanic](https://www.kaggle.com/datasets/yasserh/titanic-dataset) passenger
data to practice the rest of this week's preprocessing topics — **missing
values**, **outliers**, and **avoiding data leakage when scaling**. This is a
plain dataset (not a Kaggle competition), so it downloads with no login
required.

In [ ]:
!pip install -q kagglehub
import kagglehub
import glob

titanic_dir = kagglehub.dataset_download("yasserh/titanic-dataset")
csv_path = glob.glob(f"{titanic_dir}/**/*.csv", recursive=True)[0]
print("Using file:", csv_path)

titanic = pd.read_csv(csv_path)
print("Shape:", titanic.shape)
titanic.head()

### Checking for missing values

As covered in lecture, missing values don't mean we should just drop every
row that has one — real-world data has missing values all the time, and
dropping too aggressively throws away good data. First let's see which
columns have how much missing.

In [ ]:
print(titanic.isna().sum())
print(f"\nAge missing: {titanic['Age'].isna().mean():.1%}")
print(f"Cabin missing: {titanic['Cabin'].isna().mean():.1%}")
print(f"Embarked missing: {titanic['Embarked'].isna().mean():.1%}")

### Three different strategies for three different columns

Matching the strategies from lecture to what each column actually needs:

- **`Cabin`**: over 70% missing — at this rate, imputing is unreliable, so
  we **drop the whole column** instead.
- **`Embarked`**: only 2 missing values — fill with the **mode** (most
  common value).
- **`Age`**: about 20% missing — too much to drop, but a single global mean
  would be crude. We use the lecture's "mean within the same category" idea:
  group by `Pclass` and `Sex`, and fill with **that group's median** (median
  is less sensitive to extreme values than mean).

In [ ]:
titanic = titanic.drop(columns=["Cabin"])
print("Dropped Cabin (over 70% missing)")

embarked_mode = titanic["Embarked"].mode()[0]
titanic["Embarked"] = titanic["Embarked"].fillna(embarked_mode)
print(f"Filled Embarked missing values with the mode: '{embarked_mode}'")

titanic["Age"] = titanic.groupby(["Pclass", "Sex"])["Age"].transform(lambda s: s.fillna(s.median()))
titanic["Age"] = titanic["Age"].fillna(titanic["Age"].median())  # fallback if a group has no valid values at all
print("Filled Age missing values with the (Pclass, Sex) group median")

print("\nMissing values after cleaning:")
print(titanic.isna().sum())

### Finding outliers

Plotting `Fare` as a boxplot shows most passengers paid a modest fare, while
a handful paid a lot more. Of the methods mentioned in lecture (IQR,
Z-score, LOF, DBSCAN), we'll use the simplest: the **IQR method** — anything
outside `Q1 - 1.5*IQR` to `Q3 + 1.5*IQR` counts as an outlier.

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(titanic["Fare"])
plt.title("Fare Distribution (boxplot)")
plt.ylabel("Fare")
plt.show()

Q1, Q3 = titanic["Fare"].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = titanic[(titanic["Fare"] < lower) | (titanic["Fare"] > upper)]

print(f"IQR range: [{lower:.2f}, {upper:.2f}]")
print(f"Passengers flagged as outliers: {len(outliers)} / {len(titanic)}")
outliers[["Pclass", "Sex", "Age", "Fare"]].sort_values("Fare", ascending=False).head(10)

**Remove them, or keep them?** As lecture emphasized, an outlier isn't
automatically an error. Here, the high-fare passengers are mostly genuine
**1st-class (Pclass=1) passengers** — not a measurement mistake, just people
who paid for an expensive ticket. In cases like this, it's common to either
**leave them as-is**, or, if a model turns out to be overly sensitive to
extreme values, **cap** them at some threshold (winsorizing) instead of
deleting rows outright.

### Encoding categorical columns: ordinal vs. nominal

Applying the distinction from lecture to this dataset:

- **`Pclass`** (1st < 2nd < 3rd class) is an **ordinal** category — it's
  already numeric, and that order is meaningful, so we can **use it as-is**.
- **`Sex`** and **`Embarked`** are **nominal** categories with no order, so
  we use **one-hot encoding** like in the earlier section.

In [ ]:
cat_encoded = pd.get_dummies(titanic[["Sex", "Embarked"]])
numeric_cols = ["Age", "Fare", "SibSp", "Parch"]

X = pd.concat([titanic[["Pclass"] + numeric_cols], cat_encoded], axis=1)
y = titanic["Survived"]
X.head()

### Splitting into train / validation / test, without leaking into the scaler

Two rules from lecture, applied here:

1. **Three-way split**: divide the data into train/validation/test (a common
   ratio is 6:2:2 or 7:2:1). Validation is used during training to pick the
   best model and tune hyperparameters; test is touched exactly once, at the
   very end.
2. **Scale using training data only**: when you `fit` a `StandardScaler`, it
   must only see the **training** data. Fitting it on the full dataset (or
   anything that includes val/test) leaks a bit of "future" information into
   the mean/std it learns — that's **data leakage**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)
print(f"train/val/test sizes: {len(X_train)} / {len(X_val)} / {len(X_test)}")
print(f"ratios: {len(X_train)/len(X):.2f} / {len(X_val)/len(X):.2f} / {len(X_test)/len(X):.2f}")

In [ ]:
# Wrong way: fit the scaler on the full dataset (val/test information leaks in)
scaler_wrong = StandardScaler().fit(X[numeric_cols])

# Right way: fit only on train, then just transform() val/test
scaler_right = StandardScaler().fit(X_train[numeric_cols])

print(f"Age mean/std, fit on FULL data (leaky):   {scaler_wrong.mean_[0]:.3f} / {scaler_wrong.scale_[0]:.3f}")
print(f"Age mean/std, fit on TRAIN only (correct): {scaler_right.mean_[0]:.3f} / {scaler_right.scale_[0]:.3f}")

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler_right.transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler_right.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler_right.transform(X_test[numeric_cols])

print("\ntrain mean after scaling (should be ~0):", X_train_scaled[numeric_cols].mean().mean().round(6))

Notice the two mean/std values are (slightly) different. This dataset is
small enough that the difference is minor, but with a larger dataset, or
train/val/test sets with different distributions, that gap can actually
affect model performance — and more fundamentally, it breaks the premise
that val/test are supposed to look like "data the model hasn't seen yet."

**To recap:** handling missing values appropriately per column, flagging
outliers and deciding whether to keep or cap them, encoding ordinal vs.
nominal categories correctly, and fitting the scaler on training data
only — all of this is the **Data Preprocessing** stage from the ML
pipeline at the start of lecture. **Part 2**, right below, uses these same validation/test splits to
actually compute metrics like the confusion matrix, precision, and
recall.

## Try it yourself (Part 1)

1. **Label encoding instead of one-hot.** Try
   `cars["brand"].map({"Hyundai": 0, "Kia": 1, "Toyota": 2, "BMW": 3})` as a
   single numeric column instead of one-hot encoding. Does MAE get worse?
   Why might a linear model be misled by pretending brand is an ordered number?
2. **Add another engineered feature.** Create `mileage_per_year = mileage_km / age`
   (careful with `age == 0`) and add it to step 4 — does it help?
3. **Which single feature matters most?** Look at the model's `.coef_` after
   fitting on `X4` — which feature has the largest (absolute) coefficient?
4. **Standardizing the target?** We didn't scale `price` (the target) — why is
   that fine for regression, unlike the input features?
5. **Find outliers with Z-score instead.** Compute
   `(Fare - Fare.mean()) / Fare.std()` and flag passengers whose absolute
   value is above 3. Does it catch the same passengers as the IQR method, or
   different ones?
6. **Look again at Step 4 in the used-car example above.** It fits
   `StandardScaler` *before* `train_test_split` — isn't that the same
   leakage problem from this section? Explain why `LinearRegression`'s test
   MAE barely changes because of it, and why this ordering would matter more
   for k-NN or a neural network (Week 8).

*(이번 주는 두 개의 캡스톤이 있습니다 — Part 1(데이터 표현)과 Part 2(모델 평가) 각각에 하나씩입니다. 순서대로 진행하세요.)*

---
## 🎯 캡스톤: 우리 학교 중고거래 적정가 산출기

캠퍼스 중고거래(전공서적, 전자기기, 의류 등)의 가상 거래 기록 200건을 드립니다. 위에서 배운 **원-핫 인코딩 + feature engineering + 선형회귀**를 그대로 적용해서, 여러분이 팔고 싶은 물건의 적정가를 예측하는 모델을 직접 만들어보세요.

**확장 아이디어:** 실제 학교 커뮤니티/중고거래 게시판에서 거래 완료 글 수십 건(카테고리, 상태, 원가, 사용기간, 거래가)을 모아 `items_df`를 바꾸면, 진짜 "적정가 계산기"가 됩니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
rng = np.random.default_rng(5)
n_items = 200

categories = ["전공서적", "교양서적", "전자기기", "의류", "기타"]
conditions = ["새것같음", "양호", "사용감있음", "많이닳음"]

price_range = {"전공서적": (30000, 90000), "교양서적": (10000, 30000),
               "전자기기": (200000, 1500000), "의류": (20000, 150000), "기타": (5000, 50000)}
decay_rate = {"전공서적": 0.005, "교양서적": 0.01, "전자기기": 0.03, "의류": 0.02, "기타": 0.015}
condition_factor = {"새것같음": 0.95, "양호": 0.8, "사용감있음": 0.65, "많이닳음": 0.45}

cat_choices = rng.choice(categories, n_items)
cond_choices = rng.choice(conditions, n_items)
age_months = rng.integers(1, 48, n_items)
original_price = np.array([rng.uniform(*price_range[c]) for c in cat_choices])

decay = np.array([max(0.15, 1 - decay_rate[c] * m) for c, m in zip(cat_choices, age_months)])
cond_mult = np.array([condition_factor[c] for c in cond_choices])
sold_price = original_price * decay * cond_mult * rng.normal(1.0, 0.08, n_items)

items_df = pd.DataFrame({
    "category": cat_choices,
    "condition": cond_choices,
    "age_months": age_months,
    "original_price": original_price.round(0),
    "sold_price": sold_price.round(0),
})
items_df.head()

### 여러분의 과제

1. `category`, `condition`을 `pd.get_dummies`로 원-핫 인코딩하세요.
2. `age_months`, `original_price`와 원-핫 인코딩된 컬럼들을 합쳐 입력(X)을 만들고, `sold_price`를 정답(y)으로 하여 `LinearRegression`을 학습시키세요. (위 본문의 `evaluate()` 함수를 그대로 재사용해도 됩니다.)
3. 여러분이 실제로 팔고 싶은(또는 사고 싶은) 물건 하나를 골라 아래 `my_item`에 정보를 채우고, 학습된 모델로 적정가를 예측해보세요. **주의:** 예측할 때 입력 데이터의 컬럼 순서/이름이 학습 때와 정확히 같아야 합니다.

In [ ]:
# TODO 1: category, condition을 원-핫 인코딩하세요.


# TODO 2: age_months, original_price + 원-핫 컬럼으로 입력(X)을 구성하고 LinearRegression을 학습시키세요.


# TODO 3: 나의 물건 정보를 입력하고 적정가를 예측해보세요.
my_item = {
    "category": None,        # 예: "전자기기"
    "condition": None,       # 예: "양호"
    "age_months": None,      # 예: 10
    "original_price": None,  # 예: 800000
}

## Part 2: Model Evaluation — Breast Cancer Diagnosis Evaluator

**Theme:** Model evaluation — cross-validation & evaluation metrics

So far we've measured models with a single train/test split and a single
accuracy number. That's not enough for a real decision — especially in a
medical context, where the *kind* of mistake matters a lot: missing a
malignant tumor (false negative) is far worse than an unnecessary follow-up
test (false positive).

**Dataset:** scikit-learn's built-in breast cancer dataset — 569 patients,
30 measurements from a tumor scan, label = malignant or benign.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, roc_auc_score, precision_recall_curve,
)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print("Classes:", dict(zip(data.target_names, [0, 1])))
print("Shape:", X.shape)
print("Class balance:", np.bincount(y), "(0=malignant, 1=benign)")

## 1. A single train/test split — and why it's not enough

We build a `Pipeline` so scaling always happens *inside* the same fit/predict
step (never fit the scaler on test data — that would leak information).

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y
)
model.fit(X_train, y_train)
single_split_acc = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy on THIS ONE split: {single_split_acc:.2%}")

In [ ]:
# What if we'd used a different random split? Try several seeds.
accs = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
    accs.append(accuracy_score(yte, m.predict(Xte)))

print("Accuracy across 10 different random splits:", [f"{a:.2%}" for a in accs])
print(f"Range: {min(accs):.2%} to {max(accs):.2%} — a single split can be lucky or unlucky!")

## 2. k-fold cross-validation: a more honest estimate

Instead of one split, **k-fold cross-validation** splits the data into `k`
parts, trains on `k-1` of them and tests on the last, and rotates — so every
example gets used for testing exactly once. We report the average.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print("5-fold CV accuracy per fold:", [f"{s:.2%}" for s in cv_scores])
print(f"Mean: {cv_scores.mean():.2%}  (+/- {cv_scores.std():.2%})")

## 3. Confusion matrix: what kind of mistakes does it make?

Accuracy hides *which* mistakes happen. The confusion matrix doesn't.

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=data.target_names)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, y_pred, target_names=data.target_names))

## 4. Precision, recall, and why accuracy alone can mislead

- **Precision** — of everything we predicted "benign", how much really was benign?
- **Recall** — of everything that really was "malignant", how much did we catch?

In this task, missing a malignant tumor (a false negative for the malignant
class) is the costly mistake — so **recall on the malignant class** matters a
lot, sometimes more than overall accuracy.

In [ ]:
# malignant is class 0 in this dataset — let's look at recall for it specifically
report = classification_report(y_test, y_pred, target_names=data.target_names, output_dict=True)
print(f"Recall on malignant cases: {report['malignant']['recall']:.2%}")
print(f"Precision on malignant cases: {report['malignant']['precision']:.2%}")

## 5. ROC curve and AUC

The model doesn't just output a class — it outputs a *probability*. The ROC
curve shows the tradeoff between catching more true positives (recall) and
generating more false alarms, as we sweep the decision threshold.

In [ ]:
y_scores = model.predict_proba(X_test)[:, 1]  # probability of "benign"
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc = roc_auc_score(y_test, y_scores)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random guessing")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.show()

## Try it yourself (Part 2)

1. **Move the decision threshold.** The default threshold is 0.5. Try
   `y_pred_strict = (y_scores >= 0.9).astype(int)` (only call it "benign" if
   90%+ confident) and recompute the confusion matrix — how do precision and
   recall for "malignant" change?
2. **Precision-recall curve.** Use `precision_recall_curve(y_test, y_scores)`
   to plot precision vs. recall directly — when might this be more informative
   than the ROC curve? (Hint: think about class imbalance.)
3. **10-fold vs. 5-fold.** Re-run cross-validation with `n_splits=10` — does
   the mean accuracy change much? Does the standard deviation?
4. **A worse model on purpose.** Train a `LogisticRegression` with only 2 of
   the 30 features — how much does accuracy, recall, and AUC all drop?

---
## 🎯 캡스톤: 과제 마감 지각 위험 예측기

가상의 과제 제출 기록 250건("마감 며칠 전에 시작했는지 / 예상 소요시간 / 그 주 다른 마감 개수 / 난이도" -> "지각 여부")을 드립니다. 위에서 배운 **로지스틱 회귀 + 교차검증 + confusion matrix/정밀도·재현율**을 그대로 적용해서, 여러분의 다음 과제가 늦을 위험이 얼마나 되는지 예측하는 모델을 만들어보세요.

**확장 아이디어:** 이번 학기 실제 과제들의 (착수 시점, 예상 소요시간, 그 주 다른 과제 수, 난이도, 실제 지각 여부)를 기록해서 `assignments_df`를 바꾸면, 진짜 "내 과제 지각 위험 알리미"가 됩니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
import pandas as pd
rng = np.random.default_rng(9)
n_assignments = 250

days_before_started = np.clip(rng.exponential(3, n_assignments), 0, 14)
estimated_hours = np.clip(rng.normal(6, 3, n_assignments), 0.5, 20)
num_other_deadlines = rng.integers(0, 5, n_assignments)
difficulty = rng.integers(1, 6, n_assignments)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

logit = (
    -1.0
    - 0.55 * days_before_started
    + 0.22 * estimated_hours
    + 0.5 * num_other_deadlines
    + 0.35 * difficulty
)
late_prob = sigmoid(logit)
submitted_late = (rng.random(n_assignments) < late_prob).astype(int)

assignments_df = pd.DataFrame({
    "days_before_started": days_before_started.round(1),
    "estimated_hours": estimated_hours.round(1),
    "num_other_deadlines": num_other_deadlines,
    "difficulty": difficulty,
    "submitted_late": submitted_late,
})
print("전체 지각 비율:", assignments_df["submitted_late"].mean().round(2))
assignments_df.head()

### 여러분의 과제

1. `days_before_started`, `estimated_hours`, `num_other_deadlines`, `difficulty`를 입력(X), `submitted_late`를 정답(y)으로 하여 `StandardScaler` + `LogisticRegression` 파이프라인을 학습시키세요. (위 Part 2 본문 코드 참고)
2. `cross_val_score`로 5-fold 교차검증 평균 정확도를 출력하세요.
3. 테스트셋에 대해 confusion matrix와 `classification_report`(정밀도/재현율/F1)를 출력하세요. **지각(1)을 재현율(recall) 기준으로 살펴보면 어떤가요?** — 지각을 "놓치는 것"(재현율이 낮은 것)과 "괜히 경고하는 것"(정밀도가 낮은 것) 중 어느 쪽이 더 나쁠지 생각해보세요.
4. 아래 `my_assignment`에 다가오는 과제 정보를 입력하고, `predict_proba()`로 지각 확률을 예측해보세요.

In [ ]:
# TODO 1: StandardScaler + LogisticRegression 파이프라인을 학습시키세요.


# TODO 2: 5-fold 교차검증 평균 정확도를 출력하세요.


# TODO 3: confusion matrix와 classification_report를 출력하세요.


# TODO 4: 다가오는 과제 정보를 입력하고 지각 확률을 예측해보세요.
my_assignment = {
    "days_before_started": None,   # 예: 2 (마감 2일 전부터 시작 예정)
    "estimated_hours": None,       # 예: 8
    "num_other_deadlines": None,   # 예: 1
    "difficulty": None,            # 예: 4
}